In [1]:
#This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
#
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/sk48.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ab4f63cc/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ab4f63cc/tn36.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/dadda488/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/dadda488/m0r0.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/bp35.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/65d47d14/cn04.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/65d47d14/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-

In [2]:
# ============================================
# ARC Prize 2026 - ARC-AGI-3
# Wynt3r45 Omni-Agent Notebook
# License: CC BY 4.0 (or competition-compatible)
# ============================================

import os
import json
import time
import random
import logging
import subprocess
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests

# Optional imports; keep notebook runnable even if some extras are missing.
try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

try:
    from arc_agi import Arcade, OperationMode
except Exception:
    Arcade = None
    OperationMode = None

try:
    from arcengine import GameAction, GameState
except Exception:
    GameAction = None
    GameState = None

DOC_INDEX_URL = "https://docs.arcprize.org/llms.txt"
ROOT_URL = "https://three.arcprize.org"

API_KEY = os.getenv("ARC_API_KEY", "")
MODE = os.getenv("ARC_MODE", "auto").lower()  # auto | online | offline | toolkit
COMPETITION_MODE = os.getenv("ARC_COMPETITION_MODE", "0") == "1"

if load_dotenv:
    try:
        load_dotenv(".env")
    except Exception:
        pass

logger = logging.getLogger("wynt3r45")
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

# Kaggle paths / input scan
print("Kaggle input files:")
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# If you need to try an online connectivity check, do it safely.
def internet_available(timeout: float = 3.0) -> bool:
    try:
        requests.get(DOC_INDEX_URL, timeout=timeout)
        return True
    except Exception:
        return False

ONLINE_OK = internet_available()

def fetch_docs_index() -> str:
    if not ONLINE_OK:
        return "OFFLINE: docs index unavailable."
    try:
        r = requests.get(DOC_INDEX_URL, timeout=10)
        r.raise_for_status()
        return r.text
    except Exception as e:
        return f"DOCS_FETCH_FAILED: {e}"

docs_index = fetch_docs_index()
print("\nDocs index preview:")
print(docs_index[:1200] if isinstance(docs_index, str) else str(docs_index)[:1200])

# -------------------------------------------------------------------
# Kaggle data acquisition hooks
# -------------------------------------------------------------------
# When online and permitted, these are the commands you can use:
# !kaggle competitions download -c arc-prize-2026-arc-agi-3
# import kagglehub; kagglehub.competition_download('arc-prize-2026-arc-agi-3')
# For remote clients, prompt them to use:
# "mcp_kaggle_download_competition_data_files"
#
# In this notebook, we fall back to /kaggle/input if internet is off.

def find_local_arc_data() -> List[str]:
    files = []
    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            files.append(os.path.join(dirname, filename))
    return files

local_data_files = find_local_arc_data()

# -------------------------------------------------------------------
# ARC Toolkit / Arcade setup
# -------------------------------------------------------------------
def build_arcade(
    operation_mode: str = "normal",
    arc_api_key: Optional[str] = None,
    recordings_dir: str = "./recordings",
    environments_dir: str = "./environment_files",
):
    if Arcade is None:
        raise ImportError("arc_agi is not available in this runtime.")

    if OperationMode is None:
        raise ImportError("OperationMode is not available in this runtime.")

    mode_map = {
        "offline": OperationMode.OFFLINE,
        "online": OperationMode.ONLINE,
        "normal": OperationMode.NORMAL,
    }
    op_mode = mode_map.get(operation_mode.lower(), OperationMode.NORMAL)
    return Arcade(
        operation_mode=op_mode,
        arc_api_key=arc_api_key or os.getenv("ARC_API_KEY", ""),
        recordings_dir=recordings_dir,
        environments_dir=environments_dir,
        logger=logging.getLogger("wynt3r45"),
    )

arc = None
if Arcade is not None and OperationMode is not None:
    chosen_mode = MODE
    if chosen_mode == "auto":
        chosen_mode = "online" if (ONLINE_OK and API_KEY) else "offline"
    if chosen_mode in {"online", "offline", "normal"}:
        try:
            arc = build_arcade(chosen_mode, API_KEY)
            print(f"Arcade ready in {chosen_mode.upper()} mode")
        except Exception as e:
            print(f"Arcade init failed: {e}")

# -------------------------------------------------------------------
# Direct REST client
# -------------------------------------------------------------------
class ARCClient:
    def __init__(self, api_key: str, base_url: str = ROOT_URL):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session()
        self.session.headers.update({
            "X-API-Key": api_key,
            "Content-Type": "application/json",
            "Accept": "application/json",
        })

    def post(self, path: str, payload: Dict[str, Any]) -> requests.Response:
        r = self.session.post(f"{self.base_url}{path}", json=payload, timeout=30)
        r.raise_for_status()
        return r

    def get(self, path: str) -> requests.Response:
        r = self.session.get(f"{self.base_url}{path}", timeout=30)
        r.raise_for_status()
        return r

    def list_games(self) -> List[Dict[str, Any]]:
        return self.get("/api/games").json()

    def open_scorecard(self, source_url: Optional[str] = None, tags: Optional[List[str]] = None, opaque: Optional[Dict[str, Any]] = None) -> str:
        payload = {
            "source_url": source_url or "https://github.com/example/agent",
            "tags": tags or ["baseline", "gpt-4o"],
            "opaque": opaque or {"model": "gpt-4o", "temperature": 0.25},
        }
        response = self.session.post(
            f"{self.base_url}/api/scorecard/open",
            json=payload,
            headers={
                "X-API-Key": self.session.headers["X-API-Key"],
                "Content-Type": "application/json",
            },
            timeout=30,
        )
        print(response.text)
        response.raise_for_status()
        return response.json()["card_id"]

    def get_scorecard(self, card_id: str) -> Dict[str, Any]:
        return self.get(f"/api/scorecard/{card_id}").json()

    def get_scorecard_for_game(self, card_id: str, game_id: str) -> Dict[str, Any]:
        return self.get(f"/api/scorecard/{card_id}/{game_id}").json()

    def close_scorecard(self, card_id: str) -> Dict[str, Any]:
        return self.post("/api/scorecard/close", {"card_id": card_id}).json()

    def reset(self, game_id: str, card_id: str, guid: Optional[str] = None) -> Dict[str, Any]:
        payload = {"game_id": game_id, "card_id": card_id}
        if guid is not None:
            payload["guid"] = guid
        return self.post("/api/cmd/RESET", payload).json()

    def action(self, action_id: int, game_id: str, guid: str, reasoning: Optional[Dict[str, Any]] = None, x: Optional[int] = None, y: Optional[int] = None) -> Dict[str, Any]:
        payload = {"game_id": game_id, "guid": guid}
        if reasoning is not None:
            payload["reasoning"] = reasoning
        if action_id == 6:
            if x is None or y is None:
                raise ValueError("ACTION6 requires x and y.")
            payload["x"] = int(x)
            payload["y"] = int(y)
        return self.post(f"/api/cmd/ACTION{action_id}", payload).json()

client = ARCClient(API_KEY) if API_KEY else None

# -------------------------------------------------------------------
# Kaggle-style fallback submission building
# -------------------------------------------------------------------
@dataclass
class SubmissionRow:
    task_id: str
    prediction_json: str

def write_submission_parquet(rows: List[SubmissionRow], path: str = "submission.parquet") -> str:
    df = pd.DataFrame([asdict(r) for r in rows])
    df.to_parquet(path, index=False)
    return path

# -------------------------------------------------------------------
# Wynt3r45 persona / policy layer
# -------------------------------------------------------------------
class Wynt3r45Agent:
    def __init__(self):
        self.memory: List[Dict[str, Any]] = []
        self.replays: List[Dict[str, Any]] = []

    def log(self, payload: Dict[str, Any]) -> None:
        self.replays.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "data": payload,
        })

    def observe_frame(self, obs: Any) -> Dict[str, Any]:
        if obs is None:
            return {}
        if isinstance(obs, dict):
            return obs
        return {
            "state": getattr(obs, "state", None),
            "frame": getattr(obs, "frame", None),
            "levels_completed": getattr(obs, "levels_completed", None),
            "available_actions": [getattr(a, "name", a) for a in getattr(obs, "available_actions", [])] if hasattr(obs, "available_actions") else [],
        }

    def choose_action(self, available_actions: List[int], frame: Optional[Any] = None) -> int:
        if 7 in available_actions and len(self.memory) > 4:
            return 7
        if 6 in available_actions:
            return 6
        for preferred in [5, 4, 3, 2, 1]:
            if preferred in available_actions:
                return preferred
        return random.choice(available_actions)

    def choose_xy(self, frame: Optional[Any] = None) -> Tuple[int, int]:
        return 32, 32

    def learn(self, action: int, result: Any) -> None:
        self.memory.append({"action": action, "result": result})
        if len(self.memory) > 50:
            self.memory = self.memory[-50:]

agent = Wynt3r45Agent()

# -------------------------------------------------------------------
# Toolkit / EnvironmentWrapper path
# -------------------------------------------------------------------
def run_toolkit_episode(game_id: str, render_mode: Optional[str] = "terminal", save_recording: bool = True) -> Optional[Dict[str, Any]]:
    if arc is None:
        raise RuntimeError("Arcade is not available.")

    scorecard_id = None
    try:
        scorecard_id = arc.create_scorecard(
            source_url="https://github.com/example/agent",
            tags=["wynt3r45", "kaggle", "baseline"],
            opaque={"persona": "Wynt3r45", "mode": "toolkit"},
        )
    except Exception:
        scorecard_id = None

    env = arc.make(game_id, scorecard_id=scorecard_id, save_recording=save_recording, render_mode=render_mode)
    if env is None:
        return None

    obs = env.reset()
    steps = 0
    while obs is not None and getattr(obs, "state", None) == getattr(GameState, "NOT_FINISHED", "NOT_FINISHED") and steps < 200:
        actions = env.action_space
        action = agent.choose_action([int(a.name.replace("ACTION", "")) for a in actions if hasattr(a, "name") and a.name.startswith("ACTION")], getattr(obs, "frame", None))
        if action == 6:
            x, y = agent.choose_xy(getattr(obs, "frame", None))
            obs = env.step(GameAction.ACTION6, data={"x": x, "y": y}, reasoning={"persona": "Wynt3r45", "thought": "center probe"})
        elif action == 7:
            obs = env.step(GameAction.ACTION7, reasoning={"persona": "Wynt3r45", "thought": "undo"})
        else:
            obs = env.step(getattr(GameAction, f"ACTION{action}"), reasoning={"persona": "Wynt3r45"})
        agent.learn(action, getattr(obs, "state", None))
        agent.log({"mode": "toolkit", "action": action, "obs": str(obs)})
        steps += 1

    try:
        final_sc = arc.close_scorecard(scorecard_id) if scorecard_id else arc.get_scorecard()
    except Exception:
        final_sc = None

    return {
        "scorecard_id": scorecard_id,
        "final_scorecard": final_sc.model_dump() if hasattr(final_sc, "model_dump") else str(final_sc),
    }

# -------------------------------------------------------------------
# Direct API playtest path (full under-the-hood workflow)
# -------------------------------------------------------------------
def run_api_episode(game_id: str) -> Dict[str, Any]:
    if client is None:
        raise RuntimeError("API client requires ARC_API_KEY.")

    # Open scorecard exactly as requested
    scorecard_id = client.open_scorecard(
        source_url="https://github.com/example/agent",
        tags=["baseline", "gpt-4o"],
        opaque={"model": "gpt-4o", "temperature": 0.25},
    )

    # RESET starts the session
    state = client.reset(game_id=game_id, card_id=scorecard_id)
    guid = state.get("guid")
    current_state = state.get("state", "NOT_FINISHED")
    step_count = 0

    while current_state == "NOT_FINISHED" and step_count < 200:
        available = state.get("available_actions", [1, 2, 3, 4, 5])
        action = agent.choose_action(available, state.get("frame"))
        reasoning = {
            "persona": "Wynt3r45",
            "confidence": 0.77,
            "step": step_count,
            "notes": "hybrid reasoning loop",
        }

        if action == 6:
            x, y = agent.choose_xy(state.get("frame"))
            state = client.action(6, game_id=game_id, guid=guid, reasoning=reasoning, x=x, y=y)
        else:
            state = client.action(action, game_id=game_id, guid=guid, reasoning=reasoning)

        current_state = state.get("state", "NOT_FINISHED")
        agent.learn(action, current_state)
        agent.log({"mode": "api", "action": action, "state": current_state, "guid": guid})
        step_count += 1

        # Optional undo if you want ACTION7 support in the loop
        if current_state == "GAME_OVER" and 7 in state.get("available_actions", []):
            state = client.action(7, game_id=game_id, guid=guid, reasoning={"persona": "Wynt3r45", "thought": "undo recovery"})
            current_state = state.get("state", current_state)

    final_summary = client.close_scorecard(scorecard_id)
    return {
        "scorecard_id": scorecard_id,
        "final_state": current_state,
        "final_summary": final_summary,
    }

# -------------------------------------------------------------------
# Competition mode note
# -------------------------------------------------------------------
# Kaggle / competition mode often means:
# - run against official APIs only when allowed
# - keep one scorecard
# - preserve cookies
# - avoid noisy retries
#
# If you need competition mode with the SDK:
# from arc_agi import Arcade, OperationMode
# arc = Arcade(operation_mode=OperationMode.COMPETITION)
# and use env = arc.make(...)

# -------------------------------------------------------------------
# Listen and Serve hook
# -------------------------------------------------------------------
def listen_and_serve_stub():
    # This is a hook for the toolkit's listen_and_serve mode.
    # Example usage in a dedicated runtime:
    # arc.listen_and_serve(port=8001, competition_mode=True, save_all_recordings=True)
    pass

# -------------------------------------------------------------------
# Benchmarking / swarm hooks
# -------------------------------------------------------------------
def benchmark_runs(game_id: str, runs: int = 3) -> List[Dict[str, Any]]:
    results = []
    for i in range(runs):
        if arc is not None:
            results.append(run_toolkit_episode(game_id, render_mode=None, save_recording=True))
        elif client is not None:
            results.append(run_api_episode(game_id))
        else:
            results.append({"error": "no runtime available"})
    return results

# -------------------------------------------------------------------
# Optional Kaggle-friendly offline path
# -------------------------------------------------------------------
def build_offline_submission_from_inputs(input_dir: str = "/kaggle/input", output_path: str = "submission.parquet") -> str:
    rows = []
    for dirname, _, filenames in os.walk(input_dir):
        for filename in filenames:
            # Placeholder per-task row. Replace with your actual ARC task/prediction structure.
            rows.append(SubmissionRow(
                task_id=os.path.splitext(filename)[0],
                prediction_json=json.dumps({"source_file": os.path.join(dirname, filename), "prediction": "placeholder"})
            ))
    return write_submission_parquet(rows, output_path)

# -------------------------------------------------------------------
# Mode routing
# -------------------------------------------------------------------
def run(game_id: Optional[str] = None):
    # Prefer online API when available and an API key exists, otherwise toolkit/offline.
    if ONLINE_OK and client is not None and MODE in {"auto", "online"} and game_id:
        return run_api_episode(game_id)
    if arc is not None and game_id and MODE in {"auto", "offline", "normal"}:
        return run_toolkit_episode(game_id, render_mode="terminal", save_recording=True)
    return build_offline_submission_from_inputs()

# Example usage:
# 1) Online API: run("ls20-016295f7601e")
# 2) Toolkit local: run("ls20")
# 3) Offline Kaggle: build_offline_submission_from_inputs()

print("\nWynt3r45 notebook scaffold is ready.")
print("If you have ARC_API_KEY and online access, the API path will use docs discovery, scorecards, RESET, and ACTION1-7.")
print("If offline, it falls back to Kaggle inputs and can write submission.parquet.")

Kaggle input files:
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/sk48.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ab4f63cc/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ab4f63cc/tn36.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/dadda488/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/dadda488/m0r0.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/bp35.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/65d47d14/cn04.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/65d47d14/metadata.json
/kaggle/input/competitions/arc